# Module 1: Connecting to the ML Pipeline Debugger

Connect to a running environment, observe a broken training script, and submit a fix.

**Time:** ~15 min · **Difficulty:** Beginner · **GPU:** Not required

In [ ]:
# Install dependencies
!pip install -q openenv-core torch numpy

# Clone the environment repo (provides typed client)
!git clone --depth=1 -q https://github.com/your-username/ml-pipeline-debugger.git 2>/dev/null || true

import sys, os
sys.path.insert(0, os.path.abspath('ml-pipeline-debugger'))
print('Setup complete!')

## 1. Start the Environment Server

In a real setup you would run this in a terminal:
```bash
cd ml_pipeline_debugger
uvicorn server.app:app --host 0.0.0.0 --port 8000
```

For this notebook we start it as a background process.

In [ ]:
import subprocess, sys, time, os

env_vars = os.environ.copy()
env_vars['PYTHONPATH'] = os.path.abspath('ml-pipeline-debugger')

server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app',
     '--host', '0.0.0.0', '--port', '8000'],
    cwd='ml-pipeline-debugger',
    env=env_vars,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(4)

import urllib.request
try:
    with urllib.request.urlopen('http://localhost:8000/health', timeout=5) as r:
        print(f'Server health: {r.read().decode()}')
except Exception as e:
    print(f'Health check failed: {e}')

## 2. Connect and Call reset()

Every OpenEnv environment starts with `reset()`. It returns an observation — in our case, a broken training script.

In [ ]:
from client import MLPipelineDebuggerEnv
from models import MLDebugAction

ENV_URL = 'http://localhost:8000'

with MLPipelineDebuggerEnv(base_url=ENV_URL).sync() as env:
    result = env.reset()
    obs = result.observation

    print('=== EPISODE START ===')
    print(f'Task ID:    {obs.task_id}')
    print(f'Difficulty: {obs.difficulty}')
    print(f'Bug type:   {obs.bug_type}')
    print(f'Done:       {obs.done}')
    print(f'Reward:     {obs.reward}')
    print()
    print('--- SYMPTOM ---')
    print(obs.task_description)
    print()
    print('--- LOSS CURVE (first 5 steps) ---')
    print(obs.loss_curve[:5])
    print()
    print('--- BROKEN SCRIPT (first 20 lines) ---')
    for i, line in enumerate(obs.broken_script.split('\n')[:20], 1):
        print(f'  {i:3d}: {line}')

## 3. Call step() — Submit a Fix

After reading the broken script, submit a fix with `step()`. The grader runs your fix in a subprocess, parses the loss curve, and returns a score 0.0–1.0.

In [ ]:
# A simple fixed script — learning rate corrected to 0.01
fixed_script = """
import torch
import torch.nn as nn
torch.manual_seed(42)

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)  # FIXED
criterion = nn.MSELoss()

X = torch.randn(200, 10)
y = 3 * X[:, 0:1] + 2 * X[:, 1:2] - X[:, 2:3]

for epoch in range(100):
    pred = model(X)
    loss = criterion(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"loss:{loss.item():.6f}")
"""

with MLPipelineDebuggerEnv(base_url=ENV_URL).sync() as env:
    env.reset()   # start episode

    result = env.step(MLDebugAction(
        fixed_code=fixed_script,
        explanation='Changed lr from 10.0 to 0.01',
        task_id='task1',
    ))

    print(f'Reward:  {result.reward}')
    print(f'Done:    {result.done}')
    print()
    print('Grader feedback:')
    print(result.observation.task_description)

## 4. Call state()

`state()` returns episode metadata — useful for logging and tracking.

In [ ]:
with MLPipelineDebuggerEnv(base_url=ENV_URL).sync() as env:
    env.reset()
    state = env.state()
    print(f'Episode ID:  {state.episode_id}')
    print(f'Step count:  {state.step_count}')

## 5. Run Multiple Episodes

The environment randomly picks a task and bug variant each episode. Run several to see the variety.

In [ ]:
results_log = []

with MLPipelineDebuggerEnv(base_url=ENV_URL).sync() as env:
    for ep in range(5):
        obs = env.reset().observation
        # Submit broken script back (no fix) to see baseline score
        result = env.step(MLDebugAction(
            fixed_code=obs.broken_script,
            explanation='No fix applied',
            task_id=obs.task_id,
        ))
        results_log.append({
            'episode': ep + 1,
            'task': obs.task_id,
            'difficulty': obs.difficulty,
            'bug': obs.bug_type,
            'score': result.reward,
        })

print(f"{'Ep':>3} {'Task':>6} {'Difficulty':>10} {'Bug':>35} {'Score':>6}")
print('-' * 70)
for r in results_log:
    print(f"{r['episode']:>3} {r['task']:>6} {r['difficulty']:>10} {r['bug']:>35} {r['score']:>6.2f}")

In [ ]:
# Clean up server
server.terminate()
server.wait()
print('Server stopped.')

## Summary

| Method | What happened |
|--------|---------------|
| `reset()` | Got a broken script, loss curve, and symptom |
| `step(action)` | Submitted a fix, got a graded score 0.0–1.0 |
| `state()` | Got episode ID and step count |

Every OpenEnv environment works exactly like this.

**Next:** Module 2 — Writing LLM-based agents to automatically debug scripts.